In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
data=pd.read_csv('Amazon_Unlocked_Mobile.csv')
data.head()

,Product Name,Brand Name,Price,Rating,Reviews,Review Votes
0,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,I feel so LUCKY to have found this used (phone...,1.0
1,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,"nice phone, nice up grade from my pantach revu...",0.0
2,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,5,Very pleased,0.0
3,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,It works good but it goes slow sometimes but i...,0.0
4,"""CLEAR CLEAN ESN"" Sprint EPIC 4G Galaxy SPH-D7...",Samsung,199.99,4,Great phone to replace my lost phone. The only...,0.0


In [3]:
data['Review Votes'].value_counts()

Review Votes
0.0      278443
1.0       56930
2.0       23936
3.0       11956
4.0        6967
          ...  
165.0         1
295.0         1
364.0         1
201.0         1
335.0         1
Name: count, Length: 241, dtype: int64

In [4]:
data.isnull().sum()

Product Name        0
Brand Name      65171
Price            5933
Rating              0
Reviews            70
Review Votes    12296
dtype: int64

In [5]:
data['Brand Name']=data['Brand Name'].fillna(data['Brand Name'].mode()[0])
data['Price']=data['Price'].fillna(data['Price'].median())
data['Reviews']=data['Reviews'].fillna(data['Reviews'].mode()[0])
data['Review Votes']=data['Review Votes'].fillna(data['Review Votes'].mode()[0])

# Assumption

In [6]:
#unwanted data
import re
import nltk

data.Reviews=data.Reviews.apply(lambda x: re.sub('[^A-Za-z]',' ',x))
data.Reviews=data.Reviews.apply(lambda x: x.lower())
data.Reviews=data.Reviews.apply(lambda x: x.strip())

nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords=set(stopwords.words('english'))
data.Reviews=data.Reviews.apply(lambda x : ' '.join([word for word in x.split() if word not in stopwords]))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# data['details']=data.apply(lambda row :[row['Brand Name'],str(row['Price']),row['Rating'],row['Product Name']],axis=1)

In [8]:
data=data[['Reviews',str('Rating')]]

In [9]:
data['Rating']=data['Rating'].astype(float)

In [10]:
data=data.head(30000)

In [11]:
from sklearn.model_selection import train_test_split


In [12]:
x_train,x_test,y_train,y_test=train_test_split(data['Reviews'].to_list(),data['Rating'].to_list(),test_size=0.25,random_state=42)

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(binary=True,ngram_range=(1,2))
prepared=cv.fit_transform(x_train)
prepared_test=cv.transform(x_test)

In [14]:
# # This is used when your target is text and you want to convert into label like numbers
# # But  In That case our Target is already in numeric so we have no need to convert numeric 
# label_count=dict()
# for labels in data['Rating'].values:
#     for label in labels:
#         if label in label_count:
#             label_count[label]+=1
#         else:
#             label_count[label]=1

# # I need to do when target is text 


In [15]:
# # I need to do when target is text 
# from sklearn.preprocessing import MultiLabelBinarizer
# ml=MultiLabelBinarizer(classes=sorted(str(k) for k in label_count.keys()))
# y_train=ml.fit_transform(y_train)
# y_test=ml.transform(y_test)
                       

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
lr=LogisticRegression(solver=('lbfgs'))
one=OneVsRestClassifier(lr)


In [17]:
ready=one.fit(prepared,y_train)
prediction=ready.predict(prepared_test)

In [18]:
predict_score=ready.decision_function(prepared_test)

In [19]:
from sklearn.metrics import average_precision_score,precision_score,recall_score,accuracy_score,f1_score
def result(y_tests,predictions):
    # print("avg_pr-score :",average_precision_score(y_tests,predictions,average='micro'))
    print("f1-score:",f1_score(y_tests,predictions,average='weighted'))
    print("accuracy:",accuracy_score(y_tests,predictions))
    print("recall_score:",recall_score(y_tests,predictions,average='weighted'))
    print("precision score",precision_score(y_tests,predictions,average='weighted'))

result(y_test,prediction)

f1-score: 0.8479009871569299
accuracy: 0.8584
recall_score: 0.8584
precision score 0.8577503233781696


In [20]:
import joblib
joblib.dump(cv, 'count_vectorizer.pkl')
joblib.dump(ready, 'nlp_model.pkl')

['nlp_model.pkl']